# 05 Skew Term Structure 可视化

本模块只消费04生成的 `skew_panel`，负责整理、绘制并保存 $Skew(t,T)$；不重新拟合波动率或计算新的Skew。

## 当前模块参数值

参数默认值统一在 `00_config.ipynb` 设置；本 cell 只打印当前内核中的实际值。


In [ ]:
_module_parameter_names = ["SKEW_METHOD", "SKEW_CURVATURE_OUTPUT_PATH", "MODEL_SKEW_TAG", "SAVE_CSV", "SAVE_FIGURE", "FIGURE_FORMAT"]
print(f'05_skew_curvature.ipynb 当前参数：')
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import Image, display

plt.style.use('seaborn-v0_8-whitegrid')

## 加载上游结果

若当前内核尚未执行00和04，自动运行上游Notebook以取得配置与 `skew_panel`。

In [ ]:
_required_config = [
    'SKEW_CURVATURE_OUTPUT_PATH', 'MODEL_SKEW_TAG',
    'SAVE_CSV', 'SAVE_FIGURE', 'FIGURE_FORMAT',
]
if any(name not in globals() for name in _required_config):
    get_ipython().run_line_magic('run', './00_config.ipynb')
if 'skew_panel' not in globals():
    get_ipython().run_line_magic('run', './04_volatility_model.ipynb')


## 数据整理

保留标准字段 `TRADE_DT`、`EXPIRY`、`SKEW`，expiry完全从输入数据动态识别。

In [ ]:
# 校验并标准化04模块输出的Skew时间序列面板。
def prepare_skew_term_structure(source_panel: pd.DataFrame) -> pd.DataFrame:
    """返回按交易日和到期日排序、可直接绘图保存的三字段Skew面板。"""
    required = ['TRADE_DT', 'EXPIRY', 'SKEW']
    missing = [column for column in required if column not in source_panel.columns]
    if missing:
        raise KeyError(f'skew_panel缺少字段: {missing}')
    result = source_panel[required].copy()
    result['TRADE_DT'] = pd.to_datetime(result['TRADE_DT'], errors='coerce').dt.normalize()
    result['EXPIRY'] = pd.to_datetime(result['EXPIRY'], errors='coerce').dt.normalize()
    result['SKEW'] = pd.to_numeric(result['SKEW'], errors='coerce')
    invalid = result[required].isna().any(axis=1) | ~np.isfinite(result['SKEW'])
    if invalid.any():
        warnings.warn(f'删除{int(invalid.sum())}行无效Skew记录', RuntimeWarning)
        result = result.loc[~invalid].copy()
    duplicates = result.duplicated(['TRADE_DT', 'EXPIRY'], keep=False)
    if duplicates.any():
        raise ValueError('skew_panel中存在重复的TRADE_DT与EXPIRY组合')
    return result.sort_values(['TRADE_DT', 'EXPIRY']).reset_index(drop=True)

skew_term_structure = prepare_skew_term_structure(skew_panel)
display(skew_term_structure.head())
print(f'Rows: {len(skew_term_structure)}, Expiries: {skew_term_structure["EXPIRY"].nunique()}')

## 绘图与保存函数

In [ ]:
# 绘制所有动态识别到期日的Skew时间序列总图。
def plot_all_expiry_skew(data: pd.DataFrame, output_file) -> Path:
    """将每个expiry绘制为一条曲线并保存总览图。"""
    if data.empty:
        raise ValueError('没有可绘制的Skew数据')
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(13, 6.5))
    for expiry in sorted(data['EXPIRY'].unique()):
        sample = data.loc[data['EXPIRY'].eq(expiry)].sort_values('TRADE_DT')
        ax.plot(sample['TRADE_DT'], sample['SKEW'], marker='o', markersize=3, linewidth=1.5, label=pd.Timestamp(expiry).strftime('%Y%m'))
    ax.axhline(0.0, color='#697386', linestyle='--', linewidth=0.9, alpha=0.8)
    ax.set_title(f'Skew Term Structure - All Expiries | {MODEL_SKEW_TAG}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Trade Date')
    ax.set_ylabel('Skew')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(title='Expiry', ncol=min(4, data['EXPIRY'].nunique()), frameon=True)
    ax.grid(color='#d9e0e8', linewidth=0.7, alpha=0.8)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(output_file, dpi=170, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return output_file

# 为输入数据中每一个expiry分别绘制并保存一张Skew时间序列图。
def plot_skew_by_expiry(data: pd.DataFrame, output_dir, figure_format: str = 'png') -> dict:
    """自动循环全部expiry并返回expiry代码到图片路径的映射。"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = {}
    for expiry in sorted(data['EXPIRY'].unique()):
        expiry = pd.Timestamp(expiry)
        expiry_code = expiry.strftime('%Y%m')
        sample = data.loc[data['EXPIRY'].eq(expiry)].sort_values('TRADE_DT')
        fig, ax = plt.subplots(figsize=(10.5, 5.2))
        ax.plot(sample['TRADE_DT'], sample['SKEW'], color='#3568b8', marker='o', markersize=4, linewidth=1.7)
        ax.axhline(0.0, color='#697386', linestyle='--', linewidth=0.9, alpha=0.8)
        ax.set_title(f'Skew Time Series - {expiry_code} | {MODEL_SKEW_TAG}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Trade Date')
        ax.set_ylabel('Skew')
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(color='#d9e0e8', linewidth=0.7, alpha=0.8)
        fig.autofmt_xdate()
        fig.tight_layout()
        output_file = output_dir / f'skew_{expiry_code}.{figure_format}'
        fig.savefig(output_file, dpi=170, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        saved[expiry_code] = output_file
    return saved

# 将标准Skew期限结构表保存为UTF-8 CSV文件。
def save_skew_term_structure(data: pd.DataFrame, output_file) -> Path:
    """保存TRADE_DT、EXPIRY和SKEW三字段结果供后续模块读取。"""
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    data[['TRADE_DT', 'EXPIRY', 'SKEW']].to_csv(
        output_file, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d'
    )
    return output_file

## 执行与结果

所有图片均保存；Notebook只展示总图和按排序后的第一个expiry示例图。

In [ ]:
skew_output_path = Path(SKEW_CURVATURE_OUTPUT_PATH)
skew_figure_path = skew_output_path / 'figures'
skew_by_expiry_path = skew_figure_path / 'skew_by_expiry'
skew_data_path = skew_output_path / 'data'

skew_all_expiry_file = (
    plot_all_expiry_skew(
        skew_term_structure, skew_figure_path / f'skew_all_expiry.{FIGURE_FORMAT}'
    ) if SAVE_FIGURE else None
)
skew_by_expiry_files = (
    plot_skew_by_expiry(skew_term_structure, skew_by_expiry_path, FIGURE_FORMAT)
    if SAVE_FIGURE else {}
)
skew_term_structure_file = (
    save_skew_term_structure(skew_term_structure, skew_data_path / 'skew_term_structure.csv')
    if SAVE_CSV else None
)

print(f'Output: {skew_output_path}')
print(f'All-expiry figure: {skew_all_expiry_file}')
print(f'By-expiry figures: {len(skew_by_expiry_files)}')
print(f'Data file: {skew_term_structure_file}')

if skew_all_expiry_file is not None:
    display(Image(filename=str(skew_all_expiry_file)))
if skew_by_expiry_files:
    example_expiry = sorted(skew_by_expiry_files)[0]
    print(f'Example expiry: {example_expiry}')
    display(Image(filename=str(skew_by_expiry_files[example_expiry])))